In [1]:
import numpy as np
import pyvista as pv

from wakis import GridFIT3D

stl_file = "stl/012_LHC_WVM_6L2-316l.stl"

# re-triangulate STL
# surf = pv.read(stl_file)
# surf = surf.decimate_pro(0.5, preserve_topology=True)

In [2]:
# surf.plot(opacity=0.5, show_edges=True)

In [3]:
# # --------------------------------
# # Reference vol

import numpy as np
import pyvista as pv

from wakis import GridFIT3D

stl_file = "stl/012_LHC_WVM_6L2-316l.stl"

# Read solid
surf = pv.read(stl_file)
# surf = surf.decimate_pro(0.75)
# surf.save("stl/VSMIO_body_decimated.stl")
# Generate grid
Nx = 200
Ny = 200
Nz = 200
xmin, xmax, ymin, ymax, zmin, zmax = surf.bounds

grid = GridFIT3D(
    xmin,
    xmax,
    ymin,
    ymax,
    zmin,
    zmax,
    Nx,
    Ny,
    Nz,
    stl_solids={"shell": stl_file},
    stl_materials={"shell": "stainless steel"},
    stl_method="voxelize_rectilinear",
    verbose=2,
)

# #grid.plot_stl_mask("shell", stl_opacity=0.0)

Generating grid with 8000000 mesh cells...
    * Simulation domain bounds: 
                x:[-151.500, 101.500],
                y:[-110.150, 186.350],
                z:[7.000, 308.550]
    * Minimum cell sizes: dx=1.265e+00, dy=1.482e+00, dz=1.508e+00
Importing STL solids...


Converting polydata: 100%|██████████[00:06<00:00]
Generating binary mask: 100%|██████████[00:00<00:00]


    * STL solid shell: 551369 cells marked inside the solid.
Total grid initialization time: 10.296964168548584 s


In [7]:
from scipy.ndimage import gaussian_filter

factor = 4
Nx, Ny, Nz = grid.Nx * factor, grid.Ny * factor, grid.Nz * factor

dx = (xmax - xmin) / (Nx)
dy = (ymax - ymin) / (Ny)
dz = (zmax - zmin) / (Nz)

reference_vol = pv.ImageData(
    dimensions=(Nx, Ny, Nz),
    origin=(xmin + dx / 2, ymin + dy / 2, zmin + dz / 2),
    spacing=(dx, dy, dz),
)

surf = grid.read_stl("shell")
vox = surf.voxelize_rectilinear(reference_volume=reference_vol, progress_bar=True)
mask_ref = np.reshape(vox["mask"], (Nx, Ny, Nz), order="F")

# interpolate mask to grid Nx, Ny, Nz, computing gradient on each partial mask
dtype = np.int32
mask = np.zeros((grid.Nx, grid.Ny, grid.Nz)).astype(dtype)
for i in range(factor):
    partial = mask_ref[i::factor, i::factor, i::factor].astype(dtype) * 255 // 2
    grad = np.gradient(partial)
    partial += np.sqrt(grad[0] ** 2 + grad[1] ** 2 + grad[2] ** 2).astype(dtype)
    partial = gaussian_filter(partial, sigma=0.5).astype(dtype)
    mask += partial

mask = np.clip(mask, 0, 255).astype(dtype)
grid.grid["shell_2"] = np.reshape(mask, (grid.Nx * grid.Ny * grid.Nz))

Converting polydata: 100%|██████████[00:28<00:00]
Generating binary mask: 100%|██████████[00:00<00:00]


import matplotlib.pyplot as plt

factor = 4
S = 255 // factor
iz = grid.Nz // 2  # middle XY z-slice

fig, axes = plt.subplots(factor, 4, figsize=(18, 5 * factor))
fig.suptitle("PBA debug — XY z-slice per iteration", fontsize=13)
col_titles = ["partial (scaled)", "grad magnitude", "gauss filtered (partial+grad)", "accumulated mask"]

mask_d = np.zeros((grid.Nx, grid.Ny, grid.Nz), dtype=int)

for i in range(factor):
    # --- reproduce each intermediate matching main cell logic ---
    p0 = mask_ref[i::factor, i::factor, i::factor].astype(int) * S
    grad_d = np.gradient(p0.astype(float))
    grad_mag = np.sqrt(grad_d[0]**2 + grad_d[1]**2 + grad_d[2]**2).astype(int)
    p_filtered = gaussian_filter(p0 + grad_mag, sigma=1.0).astype(int)
    mask_d += p_filtered

    row = axes[i] if factor > 1 else axes
    slices = [p0[:, :, iz].T, grad_mag[:, :, iz].T, p_filtered[:, :, iz].T, mask_d[:, :, iz].T]
    cmaps  = ['Blues', 'hot', 'viridis', 'viridis']

    for j, (data, cmap) in enumerate(zip(slices, cmaps)):
        im = row[j].imshow(data, origin='lower', cmap=cmap, aspect='auto')
        row[j].set_title(f"i={i}  {col_titles[j]}")
        row[j].set_xlabel("x"); row[j].set_ylabel("y")
        plt.colorbar(im, ax=row[j], shrink=0.8)

plt.tight_layout()
plt.show()

print(f"S={S}, factor={factor}, iz={iz}")
print(f"p0 max: {p0.max()}, grad_mag max: {grad_mag.max()}, p_filtered max: {p_filtered.max()}, mask max: {mask_d.max()}")

import matplotlib.pyplot as plt

S = 255 // factor
ix = grid.Nx // 2  # middle YZ x-slice

fig, axes = plt.subplots(factor, 4, figsize=(18, 5 * factor))
fig.suptitle("PBA debug — YZ x-slice per iteration", fontsize=13)
col_titles = ["partial (scaled)", "grad magnitude", "gauss filtered (partial+grad)", "accumulated mask"]

mask_d = np.zeros((grid.Nx, grid.Ny, grid.Nz), dtype=int)

for i in range(factor):
    # --- reproduce each intermediate matching main cell logic ---
    p0 = mask_ref[i::factor, i::factor, i::factor].astype(int) * S
    grad_d = np.gradient(p0.astype(float))
    grad_mag = np.sqrt(grad_d[0]**2 + grad_d[1]**2 + grad_d[2]**2).astype(int)
    p_filtered = gaussian_filter(p0 + grad_mag, sigma=1.0).astype(int)
    mask_d += p_filtered

    row = axes[i] if factor > 1 else axes
    slices = [p0[ix, :, :].T, grad_mag[ix, :, :].T, p_filtered[ix, :, :].T, mask_d[ix, :, :].T]
    cmaps  = ['Blues', 'hot', 'viridis', 'viridis']

    for j, (data, cmap) in enumerate(zip(slices, cmaps)):
        im = row[j].imshow(data, origin='lower', cmap=cmap, aspect='auto')
        row[j].set_title(f"i={i}  {col_titles[j]}")
        row[j].set_xlabel("y"); row[j].set_ylabel("z")
        plt.colorbar(im, ax=row[j], shrink=0.8)

plt.tight_layout()
plt.show()

print(f"S={S}, factor={factor}, ix={ix}")
print(f"p0 max: {p0.max()}, grad_mag max: {grad_mag.max()}, p_filtered max: {p_filtered.max()}, mask max: {mask_d.max()}")

# --- Remap shell_2 to continuous PBA weights ---
# Thresholds (integer boundaries)
S      = 255 // factor          # e.g. 127 for factor=2
S_half = 255 // (factor * 2)    # e.g. 63
S_low  = 255 // (factor * 4)    # e.g. 31  (note: may exceed raw max; adjust if needed)

raw = grid.grid["shell_2"].copy()

pba_weights = np.zeros(raw.shape, dtype=float)
pba_weights[raw >= S]                        = 1.0
pba_weights[(raw >= S_half) & (raw < S)]     = 0.5
pba_weights[(raw >= S_low)  & (raw < S_half)]= 0.1  # rarely hit — S_low > S, kept for completeness

# Overwrite field with float weights
grid.grid["shell_2"] = pba_weights

print(f"Thresholds: S={S}, S_half={S_half}, S_low={S_low}")
print(f"  1.0 cells : {(pba_weights == 1.0).sum()}")
print(f"  0.5 cells : {(pba_weights == 0.5).sum()}")
print(f"  0.1 cells : {(pba_weights == 0.1).sum()}")
print(f"  0.0 cells : {(pba_weights == 0.0).sum()}")


In [8]:
del mask_ref, mask, partial, grad

In [10]:
cells_inside = grid.grid.threshold(value=255 // 16, scalars="shell_2")
stl_surf = pv.read(stl_file)

pl = pv.Plotter()
# pl.add_mesh(stl_surf, color="white", opacity=0.10, label="STL surface")
pl.add_mesh_clip_plane(
    cells_inside,
    scalars="shell_2",
    cmap="viridis",
    show_edges=True,
    label="grid.grid['shell'] cells",
    normal="x",
    widget_color="white",
)
pl.add_axes()
pl.show_grid()
pl.add_legend()
pl.enable_3_lights()
pl.show()

Widget(value='<iframe src="http://localhost:43743/index.html?ui=P_0x7f3c33a592e0_2&reconnect=auto" class="pyvi…

### Shifting the reference volume

In [ ]:
# --- shifted reference volumes: capture off-diagonal surface cells ---
# The i::factor loop already covers diagonal offsets (0,0,0) and (0.5,0.5,0.5) of the coarse cell.
# Axis-aligned half-cell shifts add the missing off-diagonal perspectives.
S = 255 // factor
dx_coarse = (xmax - xmin) / grid.Nx
dy_coarse = (ymax - ymin) / grid.Ny
dz_coarse = (zmax - zmin) / grid.Nz

axis_shifts = [
    (1.0, 0.0, 0.0),  # shift in x
    (0.0, 1.0, 0.0),  # shift in y
    (0.0, 0.0, 1.0),  # shift in z
]

for sfx, sfy, sfz in axis_shifts:
    ref_s = pv.ImageData(
        dimensions=(Nx, Ny, Nz),
        origin=(
            xmin + dx / 2 + sfx * dx_coarse,
            ymin + dy / 2 + sfy * dy_coarse,
            zmin + dz / 2 + sfz * dz_coarse,
        ),
        spacing=(dx, dy, dz),
    )
    vox_s = surf.voxelize_rectilinear(reference_volume=ref_s, progress_bar=True)
    mask_ref_s = np.reshape(vox_s["mask"], (Nx, Ny, Nz), order="F")

    for i in range(factor):
        partial_s = mask_ref_s[i::factor, i::factor, i::factor].astype(int) * S
        grad_s = np.gradient(partial_s.astype(float))
        partial_s += np.sqrt(grad_s[0] ** 2 + grad_s[1] ** 2 + grad_s[2] ** 2).astype(
            int
        )
        partial_s = gaussian_filter(partial_s, sigma=0.5).astype(int)
        grid.grid["shell_2"] += np.reshape(partial_s, (grid.Nx * grid.Ny * grid.Nz))

del mask_ref_s, partial_s, grad_s

# total accumulatons: factor (diagonal) + len(axis_shifts)*factor (shifted)
n_accum = factor + len(axis_shifts) * factor
print(
    f"Total accumulations: {n_accum}  |  shell_2 max: {grid.grid['shell_2'].max()}  |  suggested threshold: {S * n_accum // 4}"
)

In [ ]:
import matplotlib.pyplot as plt

S = 255 // factor
iz = grid.Nz // 2
ix = grid.Nx // 2
shift_labels = ["shift_x=(1,0,0)", "shift_y=(0,1,0)", "shift_z=(0,0,1)"]
dx_coarse = (xmax - xmin) / grid.Nx
dy_coarse = (ymax - ymin) / grid.Ny
dz_coarse = (zmax - zmin) / grid.Nz

fig, axes = plt.subplots(len(axis_shifts), 2, figsize=(12, 5 * len(axis_shifts)))
fig.suptitle("PBA shifted volumes — accumulated mask after each shift", fontsize=13)

axis_shifts_bench = [
    (1.0, 0.0, 0.0),
    (0.0, 1.0, 0.0),
    (0.0, 0.0, 1.0),
]

for row_idx, (sfx, sfy, sfz) in enumerate(axis_shifts_bench):
    ref_s = pv.ImageData(
        dimensions=(Nx, Ny, Nz),
        origin=(
            xmin + dx / 2 + sfx * dx_coarse,
            ymin + dy / 2 + sfy * dy_coarse,
            zmin + dz / 2 + sfz * dz_coarse,
        ),
        spacing=(dx, dy, dz),
    )
    vox_s = surf.voxelize_rectilinear(reference_volume=ref_s, progress_bar=False)
    mask_ref_s = np.reshape(vox_s["mask"], (Nx, Ny, Nz), order="F")

    mask_shift_d = np.zeros((grid.Nx, grid.Ny, grid.Nz), dtype=int)
    for i in range(factor):
        partial_s = mask_ref_s[i::factor, i::factor, i::factor].astype(int) * S
        grad_s = np.gradient(partial_s.astype(float))
        partial_s += np.sqrt(grad_s[0] ** 2 + grad_s[1] ** 2 + grad_s[2] ** 2).astype(
            int
        )
        partial_s = gaussian_filter(partial_s, sigma=0.5).astype(int)
        mask_shift_d += partial_s

    vmax = mask_shift_d.max() or 1

    # XY slice
    ax = axes[row_idx, 0]
    im = ax.imshow(
        mask_shift_d[:, :, iz].T, origin="lower", cmap="viridis", vmin=0, vmax=vmax
    )
    ax.set_title(f"{shift_labels[row_idx]} | XY  z={iz}")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    fig.colorbar(im, ax=ax)

    # YZ slice
    ax = axes[row_idx, 1]
    im = ax.imshow(
        mask_shift_d[ix, :, :].T, origin="lower", cmap="viridis", vmin=0, vmax=vmax
    )
    ax.set_title(f"{shift_labels[row_idx]} | YZ  x={ix}")
    ax.set_xlabel("y")
    ax.set_ylabel("z")
    fig.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()
print("Benchmark plot done.")